<a href="https://colab.research.google.com/github/lexie009/CBB6340/blob/main/CBB6340_Dynamic_Programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

csv_file = "egfr_myc_large_network.csv"

if csv_file not in uploaded:
    raise ValueError(f"Please upload {csv_file}")

print(f"Successfully uploaded: {csv_file}")

Saving egfr_myc_large_network.csv to egfr_myc_large_network.csv
Successfully uploaded: egfr_myc_large_network.csv


In [2]:
import math
import pandas as pd
from collections import defaultdict, deque


def load_graph(csv_path):
    """Load the CSV edge list into adjacency structures."""
    df = pd.read_csv(csv_path)

    required_columns = {"source", "target", "weight"}
    if not required_columns.issubset(df.columns):
        raise ValueError(
            "CSV must contain source, target, and weight columns."
        )

    adj = defaultdict(list)
    in_degree = defaultdict(int)
    nodes = set()

    for _, row in df.iterrows():
        if pd.isna(row["source"]) or pd.isna(row["target"]):
            raise ValueError("Source and target names cannot be missing.")

        u = str(row["source"]).strip()
        v = str(row["target"]).strip()
        w = float(row["weight"])

        if not math.isfinite(w) or w <= 0:
            raise ValueError("Edge weights must be finite and positive.")

        adj[u].append((v, w))
        in_degree[v] += 1

        nodes.add(u)
        nodes.add(v)

        # Ensure source-only nodes also have an in-degree entry.
        if u not in in_degree:
            in_degree[u] = 0

    return adj, in_degree, nodes


def get_topological_sort(nodes, adj, in_degree):
    """
    Kahn's algorithm: process nodes only after all incoming
    dependencies have been removed.

    Time complexity: O(|V| + |E|).
    """
    in_deg = dict(in_degree)

    queue = deque(
        node for node in nodes
        if in_deg[node] == 0
    )

    topo_order = []

    while queue:
        u = queue.popleft()
        topo_order.append(u)

        for v, weight in adj[u]:
            in_deg[v] -= 1

            if in_deg[v] == 0:
                queue.append(v)

    if len(topo_order) != len(nodes):
        raise ValueError(
            "Graph contains a cycle! Dynamic Programming requires a DAG."
        )

    return topo_order


def analyze_network(csv_path, source="EGFR", target="MYC"):
    """
    Compute:
      1. Minimum latency from source to target.
      2. Number of co-optimal shortest paths.
      3. Intermediate genes present on every shortest path.

    Paths whose latency differences are within EPSILON
    are treated as tied, following the assignment.
    """
    EPSILON = 1e-6

    adj, in_degree, nodes = load_graph(csv_path)

    if source not in nodes or target not in nodes:
        raise ValueError(
            f"Both {source} and {target} must appear in the graph."
        )

    # --------------------------------------------------
    # 1. Topological ordering
    # --------------------------------------------------
    topo_order = get_topological_sort(nodes, adj, in_degree)

    # --------------------------------------------------
    # 2. Forward DP
    # D[v]: shortest latency from source to v
    # C[v]: number of shortest paths from source to v
    # --------------------------------------------------
    D = {node: float("inf") for node in nodes}
    C = {node: 0 for node in nodes}

    D[source] = 0.0
    C[source] = 1

    for u in topo_order:
        # Unreachable nodes cannot contribute source paths.
        if not math.isfinite(D[u]):
            continue

        for v, weight in adj[u]:
            candidate = D[u] + weight

            if candidate < D[v] - EPSILON:
                # A shorter route replaces the previous routes.
                D[v] = candidate
                C[v] = C[u]

            elif abs(candidate - D[v]) <= EPSILON:
                # An equally short route adds more paths.
                C[v] += C[u]

    if C[target] == 0:
        raise ValueError(
            f"No directed path exists from {source} to {target}."
        )

    # --------------------------------------------------
    # 3. Build incoming-edge adjacency lists
    # --------------------------------------------------
    rev_adj = defaultdict(list)

    for u in nodes:
        for v, weight in adj[u]:
            rev_adj[v].append((u, weight))

    # --------------------------------------------------
    # 4. Reverse DP
    # R[v]: number of shortest-compatible suffix paths
    #       from v to target
    # --------------------------------------------------
    R = {node: 0 for node in nodes}
    R[target] = 1

    for v in reversed(topo_order):
        if not math.isfinite(D[v]) or R[v] == 0:
            continue

        for u, weight in rev_adj[v]:
            if (
                math.isfinite(D[u])
                and abs(D[u] + weight - D[v]) <= EPSILON
            ):
                R[u] += R[v]

    # --------------------------------------------------
    # 5. Identify intermediate bottlenecks
    # F[v]: number of source-to-target shortest paths
    #       passing through v
    # --------------------------------------------------
    F = {
        node: C[node] * R[node]
        for node in nodes
    }

    total_paths = C[target]

    bottlenecks = [
        node
        for node in nodes
        if node not in {source, target}
        and F[node] == total_paths
    ]

    return D[target], total_paths, sorted(bottlenecks)


if __name__ == "__main__":
    csv_file = "egfr_myc_large_network.csv"

    try:
        latency, paths, bottlenecks = analyze_network(
            csv_file,
            source="EGFR",
            target="MYC"
        )

        print("=== GRN DP Analysis Results ===")
        print(f"Minimum Signal Latency:  {latency:.6f} minutes")
        print(f"Co-Optimal Path Count:   {paths}")
        print(f"Intermediate Bottlenecks: {bottlenecks}")

    except (FileNotFoundError, ValueError) as error:
        print(f"Error: {error}")

=== GRN DP Analysis Results ===
Minimum Signal Latency:  7.500000 minutes
Co-Optimal Path Count:   4
Intermediate Bottlenecks: ['AKT1', 'PDK1', 'PIK3CA']
